# SmartDoc AI: Document Question Answering using RAG

## Overview

SmartDoc AI is a Retrieval-Augmented Generation (RAG) application that allows users to ask questions from their own PDF documents.

Instead of depending only on a language model's internal knowledge, the system retrieves relevant information from uploaded documents and uses that context to generate accurate responses.

This approach improves factual correctness and enables question answering on private or domain-specific data.

---

## Features

- Upload custom PDF documents
- Automatic text extraction and preprocessing
- Intelligent text chunking
- Semantic search using vector embeddings
- Context-aware answer generation with Gemini
- Interactive web interface using Streamlit

---

## Technologies Used

- Python
- LangChain
- Google Gemini API
- FAISS Vector Database
- Sentence Transformers
- Streamlit
- PyPDF

---

In [8]:
!pip install -q \
langchain==0.2.16 \
langchain-community==0.2.16 \
langchain-google-genai==1.0.10 \
faiss-cpu \
sentence-transformers \
pypdf \
streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 62.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 164.2/164.2 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.3/718.3 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is t

In [1]:
import langchain
print(langchain.__version__)

0.2.16


## Import Libraries
Import the required packages for document processing, embeddings, and answer generation.

In [2]:
import os

from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import RetrievalQA

## Upload PDF

Upload the document that will be used for question answering.

In [3]:
from google.colab import files

uploaded = files.upload()
pdf_file = list(uploaded.keys())[0]

Saving Vaibhav_Goyanka_Resume .pdf to Vaibhav_Goyanka_Resume .pdf


## Configure Gemini API

Enter your Gemini API key at runtime. The key is not stored inside the notebook.

In [16]:
from getpass import getpass
import os

os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API key: ")

Enter your Gemini API key: ··········


## Load the PDF

Read the uploaded document and extract its content.

In [17]:
loader = PyPDFLoader(pdf_file)
documents = loader.load()

print(f"Loaded {len(documents)} pages successfully.")

Loaded 1 pages successfully.


## Split the Text

Divide the document into smaller chunks for better retrieval.

In [19]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=100
)

chunks = text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks.")

Created 5 chunks.


## Create Embeddings

Convert text chunks into vector representations.

In [20]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## Build the Vector Store

Store embeddings using FAISS for efficient similarity search.

In [21]:
vector_db = FAISS.from_documents(chunks, embeddings)

print("Vector database created successfully.")

Vector database created successfully.


## Initialize Gemini

Load the language model for answer generation.

In [22]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

## Create the RAG Pipeline

Combine retrieval and generation into a single workflow.

In [23]:
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=vector_db.as_retriever(
        search_kwargs={"k": 3}
    )
)

## Ask Questions

Interact with the document using natural language queries.

In [27]:
while True:

    question = input("Ask a question (type 'exit' to stop): ")

    if question.lower() == "exit":
        break

    answer = qa_chain.run(question)

    print("\nAnswer:")
    print(answer)

Ask a question (type 'exit' to stop): What are the candidate's technical skills?

Answer:
The candidate's technical skills include:

*   **Programming Languages:** Java, Python, JavaScript (ES6), HTML5, CSS, SQL
*   **Frameworks:** React.js, Express.js, Angular
*   **Dev Tools:** Git, GitHub, Postman, REST APIs
*   **Other Technical Skills:** Data Structures and Algorithms, Database Management System, Object Oriented Programming
Ask a question (type 'exit' to stop): What projects has the candidate worked on?

Answer:
The candidate has worked on the following project:

*   **Healthcare Management System**
    *   This is a personal project implemented using ReactJS and MongoDB. It's a web application designed to provide accessible, convenient, and affordable healthcare services. It features two entities: Admin (who manages tools and nurses) and User (who can consult doctors, book nurses, and access services like oxygen cylinders and ICU beds).
Ask a question (type 'exit' to stop): What 